# Episode 3 — `buffer~` + `wave~`: the Wavetable Oscillator

⏱ About 30 minutes · Difficulty ★★☆

**Musical element this episode controls: timbre — a timbre you design yourself.**

**The equivalent Max patch:** a `buffer~` holding one cycle of a waveform, read by `phasor~` → `wave~`.

**What you will hear:**
1. a small-organ-like tone (3 harmonics stacked)
2. the same pitch getting brighter, darker or more hollow as the harmonic mix changes
3. a waveform you "draw by hand"

**By the end you can:**
- build a wavetable by stacking harmonics
- read a table with phase, and explain why interpolation is needed
- design your own timbres
- explain how a wavetable sidesteps last episode's aliasing

## Toolbox

Everything built in earlier episodes, copied as-is. Just run it; no need to re-read (the tricky lines carry `#` comments).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

sr = 44100

def play(x, gain=0.3):
    # np.clip holds anything beyond ±1 so nothing blows up
    return Audio(np.clip(gain * np.asarray(x), -1, 1), rate=sr, normalize=False)

def scope(x, title="", seconds=0.03):
    n = int(sr * seconds)                      # only plot this many samples from the start
    plt.figure(figsize=(9, 2.5))
    plt.plot(np.arange(n) / sr, np.asarray(x)[:n])
    plt.xlabel("time (s)")
    plt.title(title)
    plt.show()

freq = 110
dur = 2.0
t = np.arange(int(sr * dur)) / sr
phase = (freq * t) % 1

## 1. The idea

Last episode: phase → **formula** → waveform.

This episode: phase → **table lookup** → waveform.

The table stores **one cycle of a waveform**. Phase says "we are 30% into this cycle", so we go to the 30% position of the table and read the number there.

The benefit: the table can hold **any shape**; it need not be expressible as a formula.

## 2. Making a table: stacking harmonics

The table has `N = 2048` positions. `p` is the phase (0 to 1) of each position — built exactly like `t` in Episode 1, except it is not "which second" but "what fraction of one cycle".

A **harmonic** is a whole-number multiple of the fundamental:

- harmonic 1: completes **1** cycle across the table
- harmonic 2: completes **2** cycles
- harmonic 3: completes **3** cycles

Build each one separately first.

In [ ]:
N = 2048
p = np.arange(N) / N                 # the phase of each table position: 0, 1/2048, 2/2048 ...

h1 = np.sin(2 * np.pi * 1 * p)
h2 = np.sin(2 * np.pi * 2 * p)
h3 = np.sin(2 * np.pi * 3 * p)

fig, axes = plt.subplots(3, 1, figsize=(9, 4.5), sharex=True)
for ax, h, name in zip(axes, [h1, h2, h3], ["harmonic 1", "harmonic 2", "harmonic 3"]):
    ax.plot(h)
    ax.set_ylabel(name)
axes[-1].set_xlabel("table index (0 to 2047)")
plt.show()

Now scale each by a level and add them. It is the same operation as the chord in Episode 1 (`+` = mixing), only this time we add harmonics.

- harmonic 1 at level 1
- harmonic 2 at level 0.5
- harmonic 3 at level 0.3

In [ ]:
table = 1.0 * h1 + 0.5 * h2 + 0.3 * h3

plt.figure(figsize=(9, 3))
plt.plot(table)
plt.xlabel("table index (0 to 2047)")
plt.ylabel("amplitude")
plt.title("the table: one cycle, three harmonics added")
plt.show()

print("max amplitude:", np.max(np.abs(table)))

The maximum amplitude exceeds 1. Divide the whole table by its own maximum and the peak is exactly 1. This is **normalizing** (in Max: send `normalize 1` to `buffer~`).

### Wrap it in a function

`harmonic_amps` is a list such as `[1, 0.5, 0.3]`: the levels of harmonics 1, 2, 3 ... in order.

`for k, amp in enumerate(harmonic_amps, start=1):` means: take each level `amp` from the list in turn while `k` counts from 1 (1, 2, 3 ...). Each round is therefore "add harmonic `k` at level `amp` to the table".

In [ ]:
def make_table(harmonic_amps, N=2048):
    p = np.arange(N) / N                       # phase of every table position, 0 to 1
    table = np.zeros(N)                        # start from an empty table of zeros
    for k, amp in enumerate(harmonic_amps, start=1):
        # harmonic k: k cycles across the table, at level amp
        table += amp * np.sin(2 * np.pi * k * p)
    return table / np.max(np.abs(table))       # normalize: peak becomes 1

table = make_table([1, 0.5, 0.3])
print("length:", len(table), "  max amplitude:", np.max(np.abs(table)))

## 3. Reading the table: small numbers first

Take a table with only 8 positions, filled with numbers that are easy to recognise.

- `ph * 8` → which table position the phase points at
- `.astype(int)` → chop off the decimals to get a whole-number index

Then a key numpy move: **put a whole array of indices in the brackets and you get a whole array of values back.**

In [ ]:
tiny_table = np.array([0, 10, 20, 30, 40, 50, 60, 70])
ph = np.array([0.0, 0.25, 0.5, 0.9])

position = ph * len(tiny_table)      # phase 0..1 → table position 0..8
ix = position.astype(int)            # drop the decimals to get a usable index

print("phase       :", ph)
print("position    :", position)
print("index       :", ix)
print("values read :", tiny_table[ix])

Read it: phase 0.25 → position 2 → reads 20. Phase 0.9 → position 7.2 → chopped to 7 → reads 70.

## 4. Reading the real table

The same two lines. `idx` holds 88200 indices, so `table[idx]` is 88200 samples.

**You should hear:** a sustained small-organ-like tone at A2. Thicker than a sine, softer than a saw.

In [ ]:
idx = (phase * N).astype(int)        # which table position each sample should read
x = table[idx]                       # fetch the values at all 88200 positions at once

print("phase[:4] =", phase[:4])
print("idx[:4]   =", idx[:4])

scope(x, "table read by phase")
play(x)

The shape in the plot matches the table above, repeated 3.3 times in 0.03 seconds. **Phase sets how fast the table is read (pitch); the shape in the table sets the timbre.**

In Max this step is `index~` (no interpolation).

## 5. Interpolation

`phase * N` is usually not a whole number. Chopping the decimals means "read the nearest position to the left".

With 2048 positions the error is tiny and hard to hear. To make the problem obvious, use a sine table with **only 16 positions** on purpose.

**You should hear:** what should be a clean sine, with a clear buzzy fuzz on top.

**You should see:** a waveform made of stair steps.

In [ ]:
coarse = np.sin(2 * np.pi * np.arange(16) / 16)

stepped = coarse[(phase * 16).astype(int)]     # no interpolation: read only the nearest position on the left

scope(stepped, "16-point table, no interpolation", seconds=0.015)
play(stepped)

The fix: do not read only the left position; **read both neighbours and mix them by distance.**

Say the position is 5.3:

- left is index 5, right is index 6
- it is closer to 5 (only 0.3 away), so index 5 gets 0.7 and index 6 gets 0.3
- result = `0.7 * table[5] + 0.3 * table[6]`

In the code:

- `i0`: the left index
- `i1`: the right index. `% N` wraps to 0 at the end of the table (to the right of the last position is the first)
- `frac`: the fractional part (0.3 in the example)

This is **linear interpolation**, which is how `wave~` reads by default.

In [ ]:
def osc_table(table, phase):
    N = len(table)
    pos = phase * N                            # a position with decimals, e.g. 5.3
    i0 = np.floor(pos).astype(int) % N         # left neighbour: 5
    i1 = (i0 + 1) % N                          # right neighbour: 6 (wraps to 0 at the end)
    frac = pos - np.floor(pos)                 # the fractional part: 0.3
    # 0.7 of the left, 0.3 of the right
    return (1 - frac) * table[i0] + frac * table[i1]

smooth = osc_table(coarse, phase)

scope(smooth, "16-point table, linear interpolation", seconds=0.015)
play(smooth)

**You should hear:** the same 16-point table, with the fuzz mostly gone and close to a clean sine.

**You should see:** the stair steps have turned into straight segments.

From now on we always use `osc_table(table, phase)`. It is our `wave~`.

In [ ]:
x = osc_table(table, phase)
play(x)

## 6. Designing timbres

Swap the table: pitch stays, timbre changes.

`[1 / k for k in range(1, 17)]` is shorthand for "let `k` count from 1 to 16, compute `1 / k` each time and collect the results in a list": `[1, 1/2, 1/3, ... 1/16]`.

**You should hear four tones:**

1. `dark`: fundamental only → a plain sine
2. `bright`: 16 harmonics with levels falling as 1/k → close to a saw
3. `hollow`: odd harmonics only (1, 3, 5, 7 ...) → close to a square, hollow
4. `nasal`: upper harmonics louder than lower ones → thin and nasal

In [ ]:
tables = {
    "dark":   make_table([1]),
    "bright": make_table([1 / k for k in range(1, 17)]),
    "hollow": make_table([1, 0, 1/3, 0, 1/5, 0, 1/7, 0, 1/9, 0, 1/11]),
    "nasal":  make_table([0.2, 0.3, 0.5, 0.8, 1.0, 0.8, 0.5, 0.3]),
}

fig, axes = plt.subplots(1, 4, figsize=(12, 2.5), sharey=True)
for ax, name in zip(axes, tables):
    ax.plot(tables[name])
    ax.set_title(name)
plt.show()

for name in tables:
    print(name)
    display(play(osc_table(tables[name], phase)))

**The pattern:** more harmonics, and louder upper harmonics → brighter. Odd harmonics only → hollow.

### Try your own

Edit the numbers in the list below, re-run and listen. The list can be longer or shorter.

In [ ]:
my_table = make_table([1, 0, 0.6, 0, 0.4, 0, 0.2])

scope(osc_table(my_table, phase), "my_table")
play(osc_table(my_table, phase))

## 7. Drawing a table by hand

A table can hold any shape. Here a few "corner points" are joined by straight lines — like drawing with the mouse in Max's `waveform~`.

`np.interp(p, points_x, points_y)`: give it a few (x, y) points and it joins them with straight lines and returns the height at every position in `p`.

Note: the first and last points must have **the same** height (both 0 here), or there will be a jump where each cycle meets the next.

**You should hear:** a timbre unlike any standard waveform, fairly bright and a little rough.

In [ ]:
points_x = [0.0, 0.10, 0.35, 0.50, 0.60, 0.90, 1.0]
points_y = [0.0, 1.00, 0.20, 0.60, -0.4, -1.0, 0.0]

# join the points with straight lines and get the height at all 2048 table positions
drawn = np.interp(p, points_x, points_y)

plt.figure(figsize=(9, 3))
plt.plot(p, drawn)
plt.plot(points_x, points_y, "o")
plt.xlabel("phase")
plt.title("hand-drawn table")
plt.show()

play(osc_table(drawn, phase))

## 8. Wavetables and aliasing

Last episode: a naive 2500 Hz saw aliased because it contains infinitely high harmonics.

With a table built by `make_table`, **we decide how many harmonics it contains**. As long as the highest one stays below 22050 Hz, nothing folds back.

The rule: **number of harmonics × pitch < sr / 2**

At 2500 Hz: 22050 / 2500 = 8.8, so at most 8 harmonics.

**You should hear:**
1. the naive saw (from last episode): the high tone plus junk underneath
2. the 8-harmonic table: an equally bright high tone, but clean underneath

In [ ]:
phase_hi = (2500 * t) % 1
saw8 = make_table([1 / k for k in range(1, 9)])     # 8 harmonics: the highest is 8 × 2500 = 20000 Hz < 22050

display(play(2 * phase_hi - 1, gain=0.1))
display(play(osc_table(saw8, phase_hi), gain=0.1))

Put the other way round: the higher you play one table, the sooner it breaks this rule. Real wavetable synths (Serum included) keep several versions of a table with different harmonic counts for different registers and switch automatically. In this series we stay in the bass and mid range, so we skip that step.

## Common mistakes

| Symptom | Cause |
|---|---|
| `IndexError: index 2048 is out of bounds` | The index ran past the table. `% N` was forgotten when reading the right neighbour |
| `only integer ... arrays are valid indices` | A number with decimals went into the brackets. Use `.astype(int)` first |
| A constant ticking or fuzz in the tone | The table's start and end do not meet (different heights in a hand-drawn table) |
| Junk tones at high pitches | Too many harmonics, exceeding sr / 2 |

## Exercises

**1.** Make a table with only harmonics 1 and 5 (both at level 1). **Guess first:** will you hear one note or two?

<details><summary>Answer</summary>

```python
play(osc_table(make_table([1, 0, 0, 0, 1]), phase))
```
It sounds like two notes: the fundamental A2 plus a tone two octaves and a major third above (550 Hz). When harmonics are far apart and equally loud, the ear separates them.
</details>

**2.** Change the harmonic count of the `bright` table from 16 to 4, then to 40. **You should hear:** darker with 4, brighter and more saw-like with 40.

<details><summary>Answer</summary>

```python
for count in [4, 16, 40]:
    display(play(osc_table(make_table([1 / k for k in range(1, count + 1)]), phase)))
```
</details>

**3.** At 110 Hz, how many harmonics can a table hold without aliasing?

<details><summary>Answer</summary>

22050 / 110 = 200.45, so at most 200. In the bass range it is hardly a concern.
</details>

## Recap

| Max | Python |
|---|---|
| `buffer~` (single-cycle waveform) | `table` (an array of length 2048) |
| `normalize 1` | `table / np.max(np.abs(table))` |
| `index~` (no interpolation) | `table[idx]` |
| `wave~` (linear interpolation) | `osc_table(table, phase)` |
| drawing in `waveform~` | `np.interp(p, points_x, points_y)` |

- phase sets how fast the table is read → pitch
- the shape in the table → timbre
- you decide how many harmonics the table holds → aliasing can be avoided

**Next episode:** not one table but a stack of them, and sliding between them — Serum's WT POS knob.